# Rider Accident Analysis
### Full Pipeline: Load → Clean → Feature Eng → EDA → Theme Mapping → Risk Score → Recommendation → Insurance → Export

## 1. Import

In [ ]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## 2. Load Data

In [ ]:
df = pd.read_excel(
    "../data/2568-AccidentRider-ฟอร์ม.xlsx",
    sheet_name=1
)

print("Shape:", df.shape)
df.head()

## 3. Cleaning

### 3.1 Drop columns ที่ไม่จำเป็น (PII + admin)

In [ ]:
drop_cols = [
    "ลำดับ",
    "รหัสพนักงาน (รหัสต้องครบ7หลัก)",
    "ชื่อ",
    "สกุล",
    "ทะเบียนรถ\n(ติดกันทุกตัว)",
    "รหัสสาขา",
    "สาขาร้าน",
    "ฝ่าย",
    "เขต",
    "FC",
    "อบรมหลักสูตร คปภ.",
    "ประเภทนศ.ทวีภาคี"  # ข้อมูลน้อยเกินไป
]

df = df.drop(columns=drop_cols, errors="ignore")
print("Shape after drop:", df.shape)

### 3.2 Strip & normalize string columns

In [ ]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "<NA>": np.nan})
        )

df.isna().sum().sort_values(ascending=False).head(15)

### 3.3 Convert numeric columns

In [ ]:
age_col   = "อายุพนักงาน(ใส่ตัวเลข)"
speed_col = "ความเร็วรถขณะเกิดเหตุ \n(ใส่เฉพาะตัวเลข)"
leave_col = "วันหยุดงาน (ใส่เฉพาะตัวเลข)"

for col in [age_col, speed_col, leave_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("--- อายุ ---")
print(df[age_col].describe())
print("\n--- ความเร็ว ---")
print(df[speed_col].describe())
print("\n--- วันหยุดงาน ---")
print(df[leave_col].describe())

## 4. Feature Engineering

### 4.1 Rider & Driver Experience (เดือน)

In [ ]:
def exp_to_months(text):
    """แปลงข้อความประสบการณ์ → จำนวนเดือน (float)"""
    if pd.isna(text):
        return np.nan
    text = str(text)
    years   = re.search(r'(\d+)ปี', text)
    months  = re.search(r'(\d+)เดือน', text)
    days    = re.search(r'(\d+)วัน', text)
    weeks   = re.search(r'(\d+)สัปดาห์', text)
    y = int(years.group(1))  if years  else 0
    m = int(months.group(1)) if months else 0
    d = int(days.group(1))   if days   else 0
    w = int(weeks.group(1))  if weeks  else 0
    return y * 12 + m + d / 30 + w / 4

rider_exp_col  = "ประสบการณ์ขับรถเป็นRider (มาแล้วกี่เดือนกี่ปี)"
driver_exp_col = "ประสบการณ์ขับรถ (ขับรถมาแล้วกี่เดือนกี่ปี)"

df["rider_exp_month"]  = df[rider_exp_col].apply(exp_to_months)
df["driver_exp_month"] = df[driver_exp_col].apply(exp_to_months)

print(df[["rider_exp_month", "driver_exp_month"]].describe())

In [ ]:
df["rider_exp_group"] = pd.cut(
    df["rider_exp_month"],
    bins=[-1, 3, 12, 24, 999],
    labels=["<3m", "3-12m", "1-2y", ">2y"]
)

df["age_group"] = pd.cut(
    df[age_col],
    bins=[0, 20, 25, 30, 100],
    labels=["<20", "20-25", "26-30", ">30"]
)

print("rider_exp_group:\n", df["rider_exp_group"].value_counts(dropna=False))
print("\nage_group:\n", df["age_group"].value_counts(dropna=False))

### 4.2 Time of Incident → Hour + Period

In [ ]:
def extract_hour(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r'(\d{1,2})[:\.](\d{1,2})', str(text).strip())
    return int(match.group(1)) if match else np.nan

def get_period(hour):
    if pd.isna(hour): return np.nan
    elif hour < 6:    return "Late Night"
    elif hour < 12:   return "Morning"
    elif hour < 18:   return "Afternoon"
    else:             return "Night"

df["hour"]   = df["เวลาเกิดเหตุ"].apply(extract_hour)
df["period"] = df["hour"].apply(get_period)

print(df["period"].value_counts())

## 5. EDA (เฉพาะที่ใช้ใน Pipeline)

### 5.1 Severity Distribution (Target)

In [ ]:
severity_col = "ระดับความรุนแรงของการหยุดงาน\n"
cause_col    = "สาเหตุที่แท้จริงจากการเกิดอุบัติเหตุ (เช่น คุยโทรศัพท์ ขับรถมือเดียว รถตัดหน้า ซ้อนท้าย )"
area_col     = "พื้นที่"

print(df[severity_col].value_counts(dropna=False))

In [ ]:
# Top causes
print("Top 20 สาเหตุ:")
print(df[cause_col].value_counts().head(20))

In [ ]:
# Cause vs Severity
cause_severity = (
    pd.crosstab(
        df[cause_col],
        df[severity_col],
        normalize="index"
    ) * 100
).round(1)

cause_severity

## 6. Theme Mapping

### 6.1 CAUSE_THEME dictionary (ครบทุก cause จาก data)

In [ ]:
CAUSE_THEME = {

    # ==========================================================
    # Defensive Driving
    # ==========================================================
    "เบรกกะทันหัน": "Defensive Driving",
    "ขับรถย้อนศร": "Defensive Driving",
    "นั่งซ้อนท้าย": "Defensive Driving",
    "ซ้อนท้าย": "Defensive Driving",
    "เลี้ยวกระทันหัน": "Defensive Driving",
    "ฝ่าสัญญาณไฟจราจร": "Defensive Driving",
    "เว้นระยะห่างน้อยเกินไป": "Defensive Driving",
    "ตัดหน้า": "Defensive Driving",
    "รถตัดหน้า": "Defensive Driving",
    "พนง.ถูกชน (พนง.เป็นฝ่ายถูก)": "Defensive Driving",
    "คนเดินข้ามถนนตัดหน้ารถ": "Defensive Driving",
    "คนเดินข้ามถนน": "Defensive Driving",
    "พนง.ออกรถกะทันหัน(ไม่มองรถ/ถนน)": "Defensive Driving",
    "สตาร์ทออกตัวกะทันหัน": "Defensive Driving",
    "เปลี่ยนช่องทางกระทันหัน": "Defensive Driving",
    "ขนส่งสินค้าจำนวนมาก": "Defensive Driving",
    "แขวนสินค้ากับแฮนด์รถ": "Defensive Driving",
    "ขับรถมือเดียว": "Defensive Driving",
    "ชนท้าย": "Defensive Driving",
    "ชนรถจอดข้างทาง": "Defensive Driving",
    "ความเร็วรถของคู่กรณี": "Defensive Driving",
    "คู่กรณีฝ่าไฟแดง": "Defensive Driving",
    "คู่กรณีตัดหน้า": "Defensive Driving",
    "คนอื่นตัดหน้า": "Defensive Driving",
    "รถพ่วงตัดหน้า": "Defensive Driving",
    "รถพ่วงเบียด": "Defensive Driving",
    "รถบัสหักหลบ": "Defensive Driving",
    "รถสวนชน": "Defensive Driving",
    "พนักงานชนคู่กรณี": "Defensive Driving",
    "ออกรถกระทันหัน": "Defensive Driving",
    "เลี้ยวกะทันหัน": "Defensive Driving",

    # ==========================================================
    # Focus & Attention
    # ==========================================================
    "ใช้โทรศัพท์ขณะขับขี่": "Focus & Attention",
    "คุยโทรศัพท์": "Focus & Attention",
    "ผู้ขับขี่ผิดพลาดเอง(ตัดสินใจพลาด)": "Focus & Attention",
    "ผู้ขับขี่ผิดพลาดเอง": "Focus & Attention",
    "ตัดสินใจพลาด": "Focus & Attention",
    "ความพร้อมของพนักงาน(วูบ/หลับใน/อ่อนเพลีย)": "Focus & Attention",
    "ง่วงนอน": "Focus & Attention",
    "ขับรถหลับใน": "Focus & Attention",
    "เมา/สารเสพติด": "Focus & Attention",
    "สะพายกระเป๋ามากกว่า 1 ใบ": "Focus & Attention",
    "การทรงตัวของพนักงานขณะขับขี่": "Focus & Attention",
    "เสียสมาธิ": "Focus & Attention",
    "ประมาท": "Focus & Attention",
    "ไม่ระวัง": "Focus & Attention",
    "เผลอ": "Focus & Attention",
    "มองไม่เห็น": "Focus & Attention",
    "ไม่ดูกระจกมองข้าง": "Focus & Attention",
    "สะดุ้งตกใจ": "Focus & Attention",
    "เดินตัดหน้า": "Focus & Attention",
    "ความพร้อมของพนักงาน": "Focus & Attention",

    # ==========================================================
    # Road & Vehicle Safety
    # ==========================================================
    "เสียหลักล้ม": "Road & Vehicle Safety",
    "ชนไม้กั้น/กำแพง/วัตถุอื่นๆ": "Road & Vehicle Safety",
    "ชนอุปกรณ์ก่อสร้าง": "Road & Vehicle Safety",
    "ชนเสาไฟ": "Road & Vehicle Safety",
    "ชนต้นไม้": "Road & Vehicle Safety",
    "ชนแผงกั้น": "Road & Vehicle Safety",
    "สัตว์ตัดหน้า": "Road & Vehicle Safety",
    "สัตว์ทำร้าย": "Road & Vehicle Safety",
    "สุนัขวิ่งตัดหน้า": "Road & Vehicle Safety",
    "ถนนมืด": "Road & Vehicle Safety",
    "ถนนชำรุด": "Road & Vehicle Safety",
    "ถนนลื่น": "Road & Vehicle Safety",
    "ถนนชื้น": "Road & Vehicle Safety",
    "น้ำท่วมถนน": "Road & Vehicle Safety",
    "สะดุดอุปกรณ์ชะลอเร็วบนถนน": "Road & Vehicle Safety",
    "พื้นลื่น(เดิน)": "Road & Vehicle Safety",
    "หลุมบ่อ": "Road & Vehicle Safety",
    "ถนนแคบ": "Road & Vehicle Safety",
    "ทางโค้ง": "Road & Vehicle Safety",
    "วัตถุสิ่งกีดขวางบนถนน": "Road & Vehicle Safety",
    "สิ่งกีดขวาง": "Road & Vehicle Safety",
    "ทรายบนถนน": "Road & Vehicle Safety",
    "ฝุ่น/ทราย": "Road & Vehicle Safety",
    "กรวดบนถนน": "Road & Vehicle Safety",
    "หินบนถนน": "Road & Vehicle Safety",
    "มีน้ำมันบนถนน": "Road & Vehicle Safety",
    "น้ำมันรั่วบนถนน": "Road & Vehicle Safety",
    "ล้อรถไม่มียาง": "Road & Vehicle Safety",
    "คันเร่งค้าง": "Road & Vehicle Safety",
    "ผ้าเบรกสึก": "Road & Vehicle Safety",
    "ความพร้อมของรถ (เบรค ล้อ ไฟ)": "Road & Vehicle Safety",
    "เบรกไม่อยู่": "Road & Vehicle Safety",
    "ยางแตก": "Road & Vehicle Safety",
    "โช้คหน้าพัง": "Road & Vehicle Safety",
    "โซ่หลุด": "Road & Vehicle Safety",
    "ล้อรถติดหลุม": "Road & Vehicle Safety",
    "สะดุดฝาท่อ": "Road & Vehicle Safety",
    "ฝาท่อระบายน้ำ": "Road & Vehicle Safety",
    "ท่อระบายน้ำ": "Road & Vehicle Safety",
    "ฝาท่อน้ำ": "Road & Vehicle Safety",
    "ของหล่นจากรถ": "Road & Vehicle Safety",
    "สายไฟพาดถนน": "Road & Vehicle Safety",
    "สายไฟรัด": "Road & Vehicle Safety",
    "ไฟฟ้าดูด": "Road & Vehicle Safety",
    "รถลื่น": "Road & Vehicle Safety",
    "ลื่นล้ม": "Road & Vehicle Safety",
    "ฝนตกถนนลื่น": "Road & Vehicle Safety",
    "รถไม่มีไฟท้าย": "Road & Vehicle Safety",
    "สะดุดก้อนหิน": "Road & Vehicle Safety",

    # ==========================================================
    # Speed Awareness
    # ==========================================================
    "ขับรถเร็ว": "Speed Awareness",
    "ขับเร็ว": "Speed Awareness",
    "ขับรถเร็วเกินกำหนด": "Speed Awareness",
    "ขับรถเร็วเกินไป": "Speed Awareness",
    "เร็วเกินไป": "Speed Awareness",
    "ความเร็วสูง": "Speed Awareness",
    "ความเร็วรถเกินไป": "Speed Awareness",
    "ความเร็วรถไม่ลดลง": "Speed Awareness",
    "ความเร็วไม่ลดลง": "Speed Awareness",
    "เร่งออกตัวเร็ว": "Speed Awareness",
    "ออกตัวเร็ว": "Speed Awareness",
    "เร่งเครื่องเร็ว": "Speed Awareness",
    "เข้าโค้งเร็ว": "Speed Awareness",
    "แซงรถออกตัวเร็ว": "Speed Awareness",
    "หมุน": "Speed Awareness",
}


In [ ]:
%pip install rapidfuzz
import pandas as pd
from rapidfuzz import process

# -------------------------------
# Normalize text before mapping
# -------------------------------
NORMALIZE_DICT = {
    "เลี้ยวกะทันหัน": "เลี้ยวกระทันหัน",
    "ขับเร็ว": "ขับรถเร็ว",
    "รถตัดหน้า": "ตัดหน้า",
    "เดินตัดหน้า": "คนเดินข้ามถนนตัดหน้ารถ",
    "ฝนตกถนนลื่น": "ถนนลื่น",
    "ความพร้อมของพนักงาน": "ความพร้อมของพนักงาน(วูบ/หลับใน/อ่อนเพลีย)",
}


def normalize_cause(text):
    if pd.isna(text):
        return text

    text = str(text).strip()

    # Replace known synonyms
    return NORMALIZE_DICT.get(text, text)


# -------------------------------
# Fuzzy Matching
# -------------------------------
FUZZY_THRESHOLD = 97


def fuzzy_theme(text):

    if pd.isna(text):
        return None

    match = process.extractOne(text, CAUSE_THEME.keys())

    if match is None:
        return None

    keyword, score, _ = match

    if score >= FUZZY_THRESHOLD:
        return CAUSE_THEME[keyword]

    return None


# -------------------------------
# Main Mapping Function
# -------------------------------
def map_campaign_theme(cause):

    if pd.isna(cause):
        return None

    cause = normalize_cause(cause)

    # Exact Match
    if cause in CAUSE_THEME:
        return CAUSE_THEME[cause]

    # Fuzzy Match
    theme = fuzzy_theme(cause)

    return theme

In [ ]:
df["campaign_theme"] = df[cause_col].apply(map_campaign_theme)

unmapped = (
    df.loc[df["campaign_theme"].isna(), cause_col]
      .value_counts()
)

print("="*40)
print("Mapped   :", df["campaign_theme"].notna().sum())
print("Unmapped :", df["campaign_theme"].isna().sum())
print("="*40)

display(unmapped)


In [ ]:
unknown = (
    df.loc[df["campaign_theme"].isna(), [cause_col]]
      .drop_duplicates()
)

unknown.to_csv(
    "../data/unknown_causes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exported unknown causes.")

### 6.2 Unmapped Check — ต้องเป็น [] ก่อนไปต่อ

In [ ]:
# ตรวจสอบหลังผ่านระบบ Mapping จริง

unmapped = (
    df.loc[df["campaign_theme"].isna(), cause_col]
      .drop_duplicates()
      .sort_values()
)

if len(unmapped):
    print(f"⚠️ พบ {len(unmapped)} cause ที่ยัง map ไม่ได้")
    print(unmapped.to_string(index=False))
else:
    print("✅ ทุก cause mapped ครบแล้ว")

## 7. Risk Score Pipeline

### 7.1 Area × Theme crosstab & percentages

In [ ]:
df_causetheme = pd.crosstab(
    df[area_col],
    df["campaign_theme"]
)

# เติม column ที่ขาดให้ครบ 4 themes
for t in ["Defensive Driving", "Focus & Attention", "Road & Vehicle Safety", "Speed Awareness"]:
    if t not in df_causetheme.columns:
        df_causetheme[t] = 0

theme_pct = (
    df_causetheme
    .div(df_causetheme.sum(axis=1), axis=0)
    * 100
).round(1)

theme_pct

### 7.2 Severity ต่อ Area

In [ ]:
severity_table = (
    pd.crosstab(
        df[area_col],
        df[severity_col],
        normalize="index"
    ) * 100
).round(1)

# ดึง %หยุดงานเกิน3วัน (safe fallback)
sev_col_name = "หยุดงานเกิน3วัน"
if sev_col_name not in severity_table.columns:
    # ลองหา column ที่ชื่อใกล้เคียง
    candidates = [c for c in severity_table.columns if "เกิน" in c]
    sev_col_name = candidates[0] if candidates else severity_table.columns[-1]
    print(f"⚠️  ใช้ column '{sev_col_name}' แทน")

severity_pct = severity_table[sev_col_name]
print(severity_table)

### 7.3 Impact Map (คำนวณจากข้อมูลจริง)

In [ ]:
theme_impact = (
    pd.crosstab(
        df["campaign_theme"],
        df[severity_col],
        normalize="index"
    ) * 100
)

impact_map = theme_impact[sev_col_name].to_dict()
print("Impact Map (% หยุดงานเกิน 3 วัน ต่อ theme):")
for k, v in impact_map.items():
    print(f"  {k}: {v:.1f}%")

### 7.4 สร้าง area_summary และคำนวณ Risk Score

In [ ]:
pct_col_map = {
    "Defensive Driving": "defensive_pct",
    "Focus & Attention": "focus_pct",
    "Road & Vehicle Safety":       "road_pct",
    "Speed Awareness":   "speed_pct",
}

area_summary = pd.DataFrame({
    "severity_pct":  severity_pct,
    "defensive_pct": theme_pct["Defensive Driving"],
    "focus_pct":     theme_pct["Focus & Attention"],
    "road_pct":      theme_pct["Road & Vehicle Safety"],
    "speed_pct":     theme_pct["Speed Awareness"],
})

# Theme Score = pct × impact
for theme, pct_col in pct_col_map.items():
    score_col = pct_col.replace("_pct", "_score")
    area_summary[score_col] = (
        area_summary[pct_col] * impact_map.get(theme, 0)
    )

# Risk Score = weighted composite
area_summary["risk_score"] = (
    area_summary["severity_pct"]  * 0.5
    + area_summary["road_pct"]    * 0.3
    + area_summary["speed_pct"]   * 0.2
).round(2)

# Priority Level
area_summary["priority_level"] = pd.cut(
    area_summary["risk_score"],
    bins=[0, 20, 30, 100],
    labels=["Low", "Medium", "High"]
)

area_summary.sort_values("risk_score", ascending=False)

## 8. Recommendation Engine

### 8.1 Dominant Theme ต่อ Area

In [ ]:
score_cols = ["defensive_score", "focus_score", "road_score", "speed_score"]

pct_theme_map = {
    "defensive_pct": "Defensive Driving",
    "focus_pct":     "Focus & Attention",
    "road_pct":      "Road & Vehicle Safety",
    "speed_pct":     "Speed Awareness",
}

campaign_map = {
    "defensive_score": "Defensive Driving",
    "focus_score":     "Focus & Attention",
    "road_score":      "Road & Vehicle Safety",
    "speed_score":     "Speed Awareness",
}

# Dominant theme (by frequency %)
area_summary["dominant_theme"] = (
    area_summary[["defensive_pct", "focus_pct", "road_pct", "speed_pct"]]
    .idxmax(axis=1)
    .map(pct_theme_map)
)

# -----------------------------
# Campaign Ranking (จาก Score)
# -----------------------------
def get_campaign_priority(row):

    ranked = row[score_cols].sort_values(ascending=False)

    return pd.Series({
        "recommended_campaign": campaign_map[ranked.index[0]],
        "supporting_campaigns": " > ".join([
            campaign_map[ranked.index[1]],
            campaign_map[ranked.index[2]]
        ])
    })

area_summary[
    ["recommended_campaign", "supporting_campaigns"]
] = area_summary.apply(get_campaign_priority, axis=1)

print(
    area_summary[
        [
            "risk_score",
            "priority_level",
            "dominant_theme",
            "recommended_campaign",
            "supporting_campaigns",
        ]
    ]
)

### 8.2 Confidence

In [ ]:
sorted_scores = np.sort(area_summary[score_cols].values, axis=1)

top1 = sorted_scores[:, -1]
top2 = sorted_scores[:, -2]

area_summary["confidence"] = (
    top1 / (top1 + top2) * 100
).round(1)

area_summary[["dominant_theme", "recommended_campaign", "confidence"]]

### 8.3 Reason & Insight

In [ ]:
def make_reason(row):
    reason = (
        f"พื้นที่นี้มี Risk Score {row['risk_score']:.1f} "
        f"อยู่ในระดับ {row['priority_level']} "
        f"โดยพบอุบัติเหตุจาก '{row['dominant_theme']}' มากที่สุด "
        f"จึงควรดำเนิน Campaign '{row['recommended_campaign']}' เป็นลำดับแรก"
    )

    if row["supporting_campaigns"]:
        reason += (
            f" และเสริมด้วย "
            f"{row['supporting_campaigns']}"
        )

    return reason

def generate_insight(row):
    return (
        f"{row.name} มีระดับความเสี่ยง {row['priority_level']} "
        f"(Risk Score {row['risk_score']:.1f}) "
        f"โดยปัจจัยหลักคือ '{row['dominant_theme']}' "
        f"จึงควรเน้น '{row['recommended_campaign']}' "
        f"และเสริม '{row['supporting_campaigns']}' "
        f"เพื่อช่วยลดโอกาสเกิดอุบัติเหตุในพื้นที่"
    )

area_summary["reason"]  = area_summary.apply(
    make_reason, axis=1
)

area_summary["insight"] = area_summary.apply(generate_insight, axis=1)

area_summary["insight"].to_frame()

## 9. Insurance Recommendation

> Map severity + theme → ประเภทประกันที่แนะนำสำหรับแต่ละ rider

In [ ]:
# ตาราง mapping: theme + severity → insurance type
INSURANCE_MAP = {
    # (campaign_theme, severity_level) → insurance_recommendation
    ("Road & Vehicle Safety",       "หยุดงานเกิน3วัน"):   "PA เต็มรูปแบบ + ค่ารักษาพยาบาลสูง",
    ("Road & Vehicle Safety",       "หยุดงานไม่เกิน3วัน"): "PA มาตรฐาน + ค่ารักษาพยาบาล",
    ("Road & Vehicle Safety",       "ไม่หยุดงาน"):          "PA มาตรฐาน",
    ("Speed Awareness",   "หยุดงานเกิน3วัน"):   "PA เต็มรูปแบบ + ทุพพลภาพ",
    ("Speed Awareness",   "หยุดงานไม่เกิน3วัน"): "PA มาตรฐาน + ทุพพลภาพ",
    ("Speed Awareness",   "ไม่หยุดงาน"):          "PA พื้นฐาน",
    ("Focus & Attention", "หยุดงานเกิน3วัน"):   "PA มาตรฐาน + อุบัติเหตุกลุ่ม",
    ("Focus & Attention", "หยุดงานไม่เกิน3วัน"): "PA พื้นฐาน + อุบัติเหตุกลุ่ม",
    ("Focus & Attention", "ไม่หยุดงาน"):          "PA พื้นฐาน",
    ("Defensive Driving", "หยุดงานเกิน3วัน"):   "PA มาตรฐาน + ค่ารักษาพยาบาล",
    ("Defensive Driving", "หยุดงานไม่เกิน3วัน"): "PA พื้นฐาน",
    ("Defensive Driving", "ไม่หยุดงาน"):          "PA พื้นฐาน",
}

def get_insurance(row):
    key = (row.get("campaign_theme"), row.get(severity_col))
    return INSURANCE_MAP.get(key, "PA พื้นฐาน (default)")

df["insurance_recommendation"] = df.apply(get_insurance, axis=1)

print(df["insurance_recommendation"].value_counts())

In [ ]:
# Area-level insurance summary
AREA_INSURANCE_MAP = {

    ("Road & Vehicle Safety","High"):
        "PA เต็มรูปแบบ + ค่ารักษาพยาบาลสูง",

    ("Road & Vehicle Safety","Medium"):
        "PA มาตรฐาน + ค่ารักษาพยาบาล",

    ("Road & Vehicle Safety","Low"):
        "PA พื้นฐาน",


    ("Speed Awareness","High"):
        "PA เต็มรูปแบบ + ทุพพลภาพ",

    ("Speed Awareness","Medium"):
        "PA มาตรฐาน + ทุพพลภาพ",

    ("Speed Awareness","Low"):
        "PA พื้นฐาน",


    ("Focus & Attention","High"):
        "PA มาตรฐาน + อุบัติเหตุกลุ่ม",

    ("Focus & Attention","Medium"):
        "PA พื้นฐาน + อุบัติเหตุกลุ่ม",

    ("Focus & Attention","Low"):
        "PA พื้นฐาน",


    ("Defensive Driving","High"):
        "PA มาตรฐาน + ค่ารักษาพยาบาล",

    ("Defensive Driving","Medium"):
        "PA พื้นฐาน + ค่ารักษาพยาบาล",

    ("Defensive Driving","Low"):
        "PA พื้นฐาน",

}

def get_area_insurance(row):

    key = (
        row["recommended_campaign"],
        str(row["priority_level"])
    )

    return AREA_INSURANCE_MAP.get(
        key,
        "PA พื้นฐาน"
    )

area_summary["insurance_recommendation"] = (
    area_summary.apply(get_area_insurance, axis=1)
)

## 10. Export

In [ ]:
# ── Export area_summary ──────────────────────────────────────────────
export_cols = [
    "severity_pct",
    "defensive_pct", "focus_pct", "road_pct", "speed_pct",
    "defensive_score", "focus_score", "road_score", "speed_score",
    "risk_score",
    "priority_level",
    "dominant_theme",
    "recommended_campaign",
    "supporting_campaigns",
    "confidence",
    "reason",
    "insight",
    "insurance_recommendation",
]

out_area = area_summary[export_cols].sort_values("risk_score", ascending=False)
out_area.to_excel("area_campaign_recommendation.xlsx", index=True)
print("✅ Exported area_campaign_recommendation.xlsx")

# ── Export rider-level ───────────────────────────────────────────────
rider_export_cols = [
    area_col,
    cause_col,
    severity_col,
    "campaign_theme",
    "insurance_recommendation",
    "period",
    "rider_exp_group",
    "age_group",
]

out_rider = df[[c for c in rider_export_cols if c in df.columns]].copy()
out_rider.to_excel("rider_accident_cleaned.xlsx", index=False)
print("✅ Exported rider_accident_cleaned.xlsx")

In [ ]:
# ── Final Summary Display ────────────────────────────────────────────
print("="*70)
print("AREA RISK SUMMARY")
print("="*70)
for area, row in out_area.iterrows():
    print(f"\n📍 {area}")
    print(f"   {row['insight']}")
    print(f"   ประกัน: {row['insurance_recommendation']}")